# 01 从真实图片到 YOLO 训练标签

你已经学过 CNN、IoU、NMS、YOLOv1 的预测张量和坐标换算。这一课把已有知识连接到本地口罩检测项目。

**这次只回答：人工标注的一个框，怎样变成 YOLO 标签中的一行？**

学完后，你应该能解释一行中的五个数，并能把它还原成图片上的框。

## 1. 从分类标签走到检测标签

MNIST 中，一张图片对应一个数字类别。检测图片可能含有多个目标，每个目标都需要“类别”和“位置”。

标签是人工整理的参考答案（ground truth）；模型预测则是模型根据图片给出的结果。训练时比较这两者，才能计算损失、更新参数。

**人工标注框不等于模型已经识别出的框。** 本课图中画出的框全部来自数据集原始 XML，没有运行模型。

## 2. 看一张真正下载到本地的图片

![数据集中的人工标注框](assets/01_ground_truth.png)

图片名为 `maksssksksss0.png`，来自已下载的 Face Mask Detection 数据集。原图宽度为 $W=512$，高度为 $H=366$。

标号按 XML 中的目标顺序排列，不是按置信度排列。

| 标号 | 原始类别 | 中文含义 | 左上角 | 右下角 |
|---|---|---|---|---|
| 1 | without_mask | 未佩戴口罩 | (79, 105) | (109, 142) |
| 2 | with_mask | 正确佩戴口罩 | (185, 100) | (226, 144) |
| 3 | without_mask | 未佩戴口罩 | (325, 90) | (360, 141) |

图片坐标通常从左上角出发：$x$ 向右增大，$y$ 向下增大。这里的框标注的是人脸区域。

## 3. 类别先变成编号

本地 `configs/mask.yaml` 中，类别按如下顺序排列：

| 类别编号 | 名称 |
|---|---|
| 0 | with_mask |
| 1 | without_mask |
| 2 | mask_weared_incorrect |

因此，第一个框的类别编号是 **1**，不是 0。编号取决于类别列表的顺序，与图片中目标的先后顺序无关。

训练标签、数据配置、推理显示必须使用一致的类别映射。

## 4. 一个框有两种常见表示

XML 给的是两个角：

$$
(x_{min},y_{min},x_{max},y_{max})
$$

YOLO 检测标签保存中心点和框宽高，并相对于原图宽高归一化：

$$
(hat{x}_c,hat{y}_c,hat{w},hat{h})
$$

采用连续边界坐标时，先求像素单位的中心与宽高：

$$
x_c=\frac{x_{min}+x_{max}}{2},\qquad y_c=\frac{y_{min}+y_{max}}{2}
$$

$$
w=x_{max}-x_{min},\qquad h=y_{max}-y_{min}
$$

再分别除以对应方向的原图尺寸：

$$
\hat{x}_c=\frac{x_c}{W},\qquad \hat{y}_c=\frac{y_c}{H},\qquad \hat{w}=\frac{w}{W},\qquad \hat{h}=\frac{h}{H}
$$

横向量除以宽度，纵向量除以高度。这里的“归一化”是坐标换算，与 BatchNorm 或图片像素归一化不是同一件事。

## 5. 一步一步算第一个框

第一个框的两个角为 $(79,105)$ 和 $(109,142)$。

中心点：

$$
x_c=\frac{79+109}{2}=94,\qquad y_c=\frac{105+142}{2}=123.5
$$

框宽高：

$$
w=109-79=30,\qquad h=142-105=37
$$

归一化：

$$
\hat{x}_c=\frac{94}{512}=0.18359375
$$

$$
\hat{y}_c=\frac{123.5}{366}\approx0.33743169
$$

$$
\hat{w}=\frac{30}{512}=0.05859375,\qquad \hat{h}=\frac{37}{366}\approx0.10109290
$$

加上类别编号，保留六位小数：

```text
1 0.183594 0.337432 0.058594 0.101093
```

读成一句话：这是一个未戴口罩的人脸，中心在全图宽度约 18.36%、高度约 33.74% 的位置，框宽约占全图 5.86%，框高约占 10.11%。

## 6. 三个目标，就有三行标签

按本课统一采用的连续边界公式，整张图片的标签为：

```text
1 0.183594 0.337432 0.058594 0.101093
0 0.401367 0.333333 0.080078 0.120219
1 0.668945 0.315574 0.068359 0.139344
```

每一行的顺序都是：

```text
class_id x_center y_center width height
```

图片和标签文件通过相同的文件名主体对应：

```text
images/maksssksksss0.png
labels/maksssksksss0.txt
```

标签没有置信度列。它表达参考答案；推理结果才会带模型给出的置信度。

这里展示的是教学换算结果，没有写入正式训练数据目录。

## 7. 与上一课 YOLOv1 的关键区别

你刚学过 YOLOv1 的格内偏移，所以这里最容易混淆。

| 对比项 | 本课 YOLO 标签文件 | 上一课 YOLOv1 预测中的中心坐标 |
|---|---|---|
| 表达的内容 | 人工标注的真实框 | 模型输出中待解码的框 |
| 中心坐标的参照 | 整张图片 | 目标中心所在网格 |
| 是否要先加网格编号 | 不需要 | 解码为全图位置时需要 |

**标签里的 0.183594，直接表示全图宽度的约 18.36%，不是某个格子内部的位置。**

训练程序之后会读取这些全图标签，再根据自己的规则分配到相应预测位置。本课不展开目标分配和网络输出的解码。

## 8. 反过来检查：标签能不能画回原图

拿到归一化的中心坐标和宽高，先乘回原图尺寸：

$$
x_c=\hat{x}_cW,\quad y_c=\hat{y}_cH,\quad w=\hat{w}W,\quad h=\hat{h}H
$$

再求角点：

$$
x_{min}=x_c-\frac{w}{2},\quad y_{min}=y_c-\frac{h}{2}
$$

$$
x_{max}=x_c+\frac{w}{2},\quad y_{max}=y_c+\frac{h}{2}
$$

使用未舍入的数值，可以还原到 $(79,105,109,142)$；保留小数位后的结果会有很小的舍入误差。

准备正式数据时，抽取图片、画回标签，是检查类别和坐标有没有转换正确的直接方法。

## 9. 本地旧脚本里有一个需要核对的细节

当前项目的 `scripts/prepare_data.py` 在求中心点时，额外减去了 1，再除以图片宽高。这会使中心点比本课公式向左、向上各移动 1 个像素，框宽高不变。

这种差异涉及原始标注的坐标起点约定。不能把“减 1”当成 YOLO 归一化公式的一部分，也不能在没确认约定时盲目改动。

本课明确使用连续边界坐标公式。正式转换前，我们会核对原始 XML 的约定和实际图像，再统一转换方式并可视化检查。本次没有运行或修改该脚本。

## 10. 这一课的自检

先尝试自己回答，再展开参考答案。

1. 这张图片为什么需要三行标签？
2. 第一行的 1 表示什么？为什么它不是置信度？
3. 如果图片等比例放大两倍，框也同步放大，归一化标签是否改变？
4. 第二个框的类别编号、像素中心、像素宽高分别是多少？
5. 标签里的中心坐标，是否需要加上网格编号才能还原到原图？

<details>
<summary>参考答案</summary>

1. XML 中有三个被标注的目标，每个目标各占一行。
2. 它是类别编号，1 对应 without_mask；标签里没有置信度列。
3. 不改变。位置和尺寸的分子、分母同时放大两倍，比例保持不变。
4. 类别编号 0；中心为 (205.5, 122)；宽高为 (41, 44)。
5. 不需要。它们已经是相对于整张图片的归一化中心坐标。

</details>

下一课再看：怎样组织训练集和验证集，以及为什么不能让两者包含相同或高度相似的图片。

## 数据与代码依据

- 图片和 XML：本地项目 `datasets/archive.zip` 中的 `images/maksssksksss0.png` 与 `annotations/maksssksksss0.xml`。
- 类别映射：本地项目 `configs/mask.yaml`。
- 旧转换脚本：本地项目 `scripts/prepare_data.py`。
- 数据集来源：[Kaggle Face Mask Detection](https://www.kaggle.com/datasets/andrewmvd/face-mask-detection)。这是下载来源记录，本课数值直接从本地 XML 读取。

本课只新增学习笔记与一张图片的教学素材，没有准备完整训练集，没有开始训练，也没有修改部署程序或权重。